In [1]:
import os
import pandas as pd
import numpy as np
import boto3
from tqdm import tqdm
#from passwords import *
import matplotlib.pyplot as plt
pd.options.mode.chained_assignment = None # silence warning
import warnings
warnings.filterwarnings('ignore')
import seaborn as sns

### Functions

In [2]:
# interval to string
def interval_to_string(interval):
    flt_left = interval.left
    flt_right = interval.right
    # make string
    str_interval = f'{flt_left:0.3f} - {flt_right:0.3f}'
    return str_interval

In [3]:
# plot distribution
def plot_distribution(ser_values, str_sername, str_dirname_output, str_filename):
    # get min/max
    flt_min = ser_values.min()
    flt_max = ser_values.max()
    flt_mean = ser_values.mean()
    flt_std = ser_values.std()
    flt_mdn = ser_values.median()
    # show distribution
    fig, ax = plt.subplots(figsize=(9,5))
    str_title = f"""
    Distribution of {str_sername} - Test
    N = {len(ser_values)}; Min = {flt_min:0.2f}; Max = {flt_max:0.2f}; Mean = {flt_mean:0.2f}; Std = {flt_std:0.2f}; Mdn = {flt_mdn:0.2f}
    """
    ax.set_title(str_title)
    sns.distplot(list(ser_values), ax=ax)
    # save
    plt.savefig(f'{str_dirname_output}/{str_filename}', bbox_inches='tight')
    # close
    plt.close()

In [4]:
# non-numeric lift plots
def non_numeric_lift_plot(col, df, str_dirname_output, bool_low_unique):
    list_str_cols = [
        col,
        'AD',
        'PD',
        'LGD',
        'ECNL',
    ]
    # group by col
    df_grouped = df[list_str_cols].groupby(col, as_index=False).mean()
    # rm col
    list_str_cols = [str_col for str_col in list_str_cols if str_col != col]
    # ax
    fig, ax = plt.subplots(nrows=len(list_str_cols), figsize=(9,18))
    for a, str_col in enumerate(list_str_cols):
        # sort
        if bool_low_unique:
            df_grouped.sort_values(by=col, ascending=True, inplace=True)
        else:
            df_grouped.sort_values(by=str_col, ascending=True, inplace=True)
        # title
        ax[a].set_title(f'{str_col} by {col}')
        # plot
        x = df_grouped[col]
        y = df_grouped[str_col]
        ax[a].plot(x, y)
        # rotate
        ax[a].set_xticklabels(x, rotation=90)
    # fix overlap
    plt.tight_layout()
    # save
    str_filename = f'plt_lift_{col}.png'
    str_local_path = f'{str_dirname_output}/{str_filename}'
    plt.savefig(str_local_path, bbox_inches='tight')
    # close
    plt.close()

In [5]:
# get quantiles
def get_col_quantiles(df, col, int_n_quantiles=20):
    # create tmp df
    list_str_cols = [
        col,
        'AD',
        'PD',
        'LGD',
        'ECNL',
    ]
    df_tmp = df[list_str_cols]
    # get decile
    df_tmp['quantile'] = pd.qcut(df_tmp[col], int_n_quantiles, labels=None, duplicates='drop')
    # get rank
    df_tmp['quantile_rank'] = pd.qcut(df_tmp[col], int_n_quantiles, labels=False, duplicates='drop')
    # group
    df_grouped = df_tmp.groupby('quantile_rank', as_index=False).agg({
        'AD': 'mean',
        'PD': 'mean',
        'LGD': 'mean',
        'ECNL': 'mean',
        'quantile': 'first',
    })
    # convert to string
    df_grouped['quantile'] = df_grouped['quantile'].apply(interval_to_string)
    # return
    return df_grouped

In [6]:
# make plots
def make_plots(df_grouped, col, str_dirname_output):
    list_str_yhat = [
        'AD',
        'PD',
        'LGD',
        'ECNL',
    ]
    # get nrows
    int_nrows = len(list_str_yhat)
    fig, ax = plt.subplots(nrows=int_nrows, ncols=1, figsize=(9, 18))
    # make plots
    for a, str_yhat in enumerate(list_str_yhat):
        # title
        str_title = f'{str_yhat} by Quantile for {col}'
        ax[a].set_title(str_title)
        # plot
        x = df_grouped['quantile_rank']
        y = df_grouped[str_yhat]
        ax[a].plot(x, y)
        # xlabels
        ax[a].set_xticklabels(df_grouped['quantile'], rotation=90)
    # fix overlap
    plt.tight_layout()
    # save
    str_filename = f'plt_{col}.png'
    str_local_path = f'{str_dirname_output}/{str_filename}'
    plt.savefig(str_local_path, bbox_inches='tight')
    # close
    plt.close()

In [7]:
# numeric high cardinality
def numeric_high_cardinality_lift_plot(df, col, int_n_quantiles, str_dirname_output):
    # get quantiles
    df_grouped = get_col_quantiles(
        df=df, 
        col=col, 
        int_n_quantiles=int_n_quantiles,
    )
    # make plot
    make_plots(
        df_grouped=df_grouped, 
        col=col, 
        str_dirname_output=str_dirname_output,
    )

In [8]:
# upload to s3
def upload_to_s3(str_local_path, str_bucket_path, str_project):
    boto3.resource('s3').Bucket(str_project).Object(str_bucket_path).upload_file(str_local_path)

### Constants

In [9]:
str_project = '20231010-gen-xii'
str_dirname_output = './output'
int_threshold = 100
int_n_quantiles = 20
str_variant = 'noPTImodel10'

### Output directory

In [10]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Variant directory

In [11]:
try:
    os.mkdir(f'{str_dirname_output}/{str_variant}')
except:
    pass

### Distplots directory

In [12]:
try:
    os.mkdir(f'{str_dirname_output}/{str_variant}/distplots')
except:
    pass

### Lift plots directory

In [13]:
try:
    os.mkdir(f'{str_dirname_output}/{str_variant}/liftplots')
except:
    pass

### Import data

In [14]:
%%time

str_filename = 'df_pd_pre_with_yhats.gzip'
str_uri = f's3://{str_project}/ad_hoc/get_predictions/{str_variant}/{str_filename}'
df = pd.read_parquet(str_uri)

# subset
df = df[df['data_set'] == 'test'].copy()

# show
df

CPU times: user 7.7 s, sys: 8.23 s, total: 15.9 s
Wall time: 4.09 s


,bigaccountid__app,linkf060__tu,linkf045__tu,linkf079__tu,linkf185__tu,linkf105__tu,linkf195__tu,linkf193__tu,linkb012__tu,linkf032__tu,...,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age,yhat_ad,yhat_pricing_pd,yhat_pricing_lgd
12401,3932700.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,1.165459,10,4,0.09,1.282051,1.0,5.879452,0.265997,0.208567,0.619272
26947,3932720.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,1.165459,10,4,0.06,1.322581,3.0,6.441096,0.466850,0.147658,0.638730
26946,3932720.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,1.165459,10,4,0.06,1.322581,3.0,6.441096,0.475868,0.158812,0.638730
16514,3932724.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,1.165459,10,4,0.15,1.195652,1.0,6.736986,0.264749,0.203020,0.607942
16515,3932724.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,1.165459,10,4,0.15,1.195652,1.0,6.736986,0.259356,0.191142,0.607942
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7864,4812498.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,1.144717,12,4,0.09,1.548387,2.0,10.010959,0.336901,0.145701,0.592475
7863,4812498.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,1.144717,12,4,0.09,1.548387,2.0,10.010959,0.343272,0.153010,0.592475
20360,4812503.0,0.0,0.0,0.0,1182.0,1182.0,1182.0,1182.0,0.0,n,...,1.144717,12,4,0.12,1.044444,-1.0,0.208219,0.341951,0.118688,0.689688
20361,4812503.0,0.0,0.0,0.0,1182.0,1182.0,1182.0,1182.0,0.0,n,...,1.144717,12,4,0.12,1.044444,-1.0,0.208219,0.332062,0.104094,0.689688


In [15]:
# group to get the mean pd and lgd by account
df_tmp = df.groupby(by='bigaccountid__app', as_index=False).agg({
    'yhat_ad': 'mean',
    'yhat_pricing_pd': 'mean',
    'yhat_pricing_lgd': 'mean',
})
# rename
dict_rename = {
    'yhat_ad': 'AD',
    'yhat_pricing_pd': 'PD',
    'yhat_pricing_lgd': 'LGD',
}
df_tmp.rename(columns=dict_rename, inplace=True)
# get ecnl
if str_variant =='noPTImodel10':
    df_tmp['ECNL'] = (df_tmp['PD'] * df_tmp['LGD']) * 2.36
else:
    df_tmp['ECNL'] = (df_tmp['PD'] * df_tmp['LGD'])
# show
df_tmp

,bigaccountid__app,AD,PD,LGD,ECNL
0,3932700.0,0.265997,0.208567,0.619272,0.304816
1,3932720.0,0.471359,0.153235,0.638730,0.230987
2,3932724.0,0.262052,0.197081,0.607942,0.282761
3,3932730.0,0.528426,0.505687,0.620471,0.740485
4,3932752.0,0.603283,0.355160,0.673531,0.564540
...,...,...,...,...,...
27802,4812483.0,0.188268,0.034249,0.579950,0.046877
27803,4812497.0,0.599719,0.354935,0.597389,0.500401
27804,4812498.0,0.340086,0.149356,0.592475,0.208835
27805,4812503.0,0.337006,0.111391,0.689688,0.181307


### Make distribution plots of the predictions

In [16]:
list_str_colname = [
    'AD',
    'PD',
    'LGD',
    'ECNL',
]

for col in tqdm(list_str_colname):
    plot_distribution(
        ser_values=df_tmp[col], 
        str_sername=col, 
        str_dirname_output=f'{str_dirname_output}/{str_variant}/distplots', 
        str_filename=f'plt_dist_{col}.png',
    )

100%|██████████| 4/4 [00:01<00:00,  2.12it/s]


### Make lift plots

In [17]:
# get app cols
list_cols = [
    'ENG-loan_to_value',
    'ENG-payment_to_income',
    'ENG-dealership_age',
    'fltgrossmonthly__income_sum',
]
for col in df.columns:
    if ('__app' in col) and (col not in list_cols):
        list_cols.append(col)

# rm some cols
list_cols_rm = [
    'strcity__app',
    'dealercity__app',
    'strzipcode__app',
    'bitapproved__app',
    'bitsystemdecline__app',
    'approvaldate__app',
    'bitfunded__app',
    'fundeddate__app',
    'dtmstampcreation__app',
    'dtmapproved__app',
    'dtmdeclined__app',
    'defaultdate__app',
    'chargeoffdate__app',
    'defaultamount__app',
    'chargeoffamount__app',
    'bigdealerid__app',
    'dealerzip__app',
    'uniqueid__app',
    'bigaccountid__app',
    'bigdebtorid__app',
    'applicationdate__app',
    'dtmfunded__app',
    'dealerstampcreation__app',
]
list_cols = [col for col in list_cols if col not in list_cols_rm]

# add additional columns
list_cols_additional = [
]

# combine lists
list_cols = list_cols + list_cols_additional
# rm dups
list_cols = list(dict.fromkeys(list_cols))

print(f'Creating lift plots for {len(list_cols)} columns:')
for a, col in enumerate(list_cols):
    print(f'{a+1} - {col}')

Creating lift plots for 47 columns:
1 - ENG-loan_to_value
2 - ENG-payment_to_income
3 - ENG-dealership_age
4 - fltgrossmonthly__income_sum
5 - fltapprovedloantovalue__app
6 - fltapprovedservicecontract__app
7 - fltservicecontract__app
8 - fltinsuredlifeamount__app
9 - fltgapinsurance__app
10 - fltapproveddowntotal__app
11 - strname__app
12 - intservicecontractmileageaddon__app
13 - strdealershiptrackertype__app
14 - bigdealertypeid__app
15 - dealerstate__app
16 - intterm__app
17 - bitrouteone__app
18 - bitservicecontract__app
19 - bitmaintenanceagreement__app
20 - fltallowance__app
21 - fltinsureddisabilityamount__app
22 - fltdowncash__app
23 - bitdealerapplicantsamezip__app
24 - bitdealerapplicantsamecity__app
25 - bitdealerapplicantsamestate__app
26 - bitdebtor__app
27 - bitdealertrack__app
28 - intservicecontractmileage__app
29 - intservicecontractterm__app
30 - intopenbktype__app
31 - fltlicensefee__app
32 - fltdocumentfee__app
33 - dti__app
34 - strvehicletype__app
35 - bitgap__ap

In [18]:
# rename
dict_rename = {
    'yhat_ad': 'AD',
    'yhat_pricing_pd': 'PD',
    'yhat_pricing_lgd': 'LGD',
}
df.rename(columns=dict_rename, inplace=True)
# get ecnl
df['ECNL'] = df['AD'] * df['PD']
# show
df

,bigaccountid__app,linkf060__tu,linkf045__tu,linkf079__tu,linkf185__tu,linkf105__tu,linkf195__tu,linkf193__tu,linkb012__tu,linkf032__tu,...,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age,AD,PD,LGD,ECNL
12401,3932700.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,10,4,0.09,1.282051,1.0,5.879452,0.265997,0.208567,0.619272,0.055478
26947,3932720.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,10,4,0.06,1.322581,3.0,6.441096,0.466850,0.147658,0.638730,0.068934
26946,3932720.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,10,4,0.06,1.322581,3.0,6.441096,0.475868,0.158812,0.638730,0.075574
16514,3932724.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,10,4,0.15,1.195652,1.0,6.736986,0.264749,0.203020,0.607942,0.053749
16515,3932724.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,10,4,0.15,1.195652,1.0,6.736986,0.259356,0.191142,0.607942,0.049574
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7864,4812498.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,12,4,0.09,1.548387,2.0,10.010959,0.336901,0.145701,0.592475,0.049087
7863,4812498.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,12,4,0.09,1.548387,2.0,10.010959,0.343272,0.153010,0.592475,0.052524
20360,4812503.0,0.0,0.0,0.0,1182.0,1182.0,1182.0,1182.0,0.0,n,...,12,4,0.12,1.044444,-1.0,0.208219,0.341951,0.118688,0.689688,0.040585
20361,4812503.0,0.0,0.0,0.0,1182.0,1182.0,1182.0,1182.0,0.0,n,...,12,4,0.12,1.044444,-1.0,0.208219,0.332062,0.104094,0.689688,0.034565


In [19]:
# iterate and make plots
list_nunique_1 = []
for col in tqdm(list_cols):
    # dirname output
    str_dirname_plots = f'{str_dirname_output}/{str_variant}/liftplots'
    # get dtype
    str_dtype = df[col].dtype
    # get number unique
    int_nunique = len(df[col].unique())
    # logic
    if int_nunique == 1:
        list_nunique_1.append(col)
    elif str_dtype not in ['int64','float64']: # if non-numeric
        # non-numeric lift plot
        non_numeric_lift_plot(
            col=col, 
            df=df, 
            str_dirname_output=str_dirname_plots,
            bool_low_unique=False,
        )
    elif (str_dtype in ['int64','float64']) and (int_nunique <= int_threshold): # numeric but there is a low number of unique
        # non-numeric lift plot
        non_numeric_lift_plot(
            col=col, 
            df=df, 
            str_dirname_output=str_dirname_plots,
            bool_low_unique=True,
        )
    else:
        # numeric high cardinality
        numeric_high_cardinality_lift_plot(
            df=df, 
            col=col, 
            int_n_quantiles=int_n_quantiles, 
            str_dirname_output=str_dirname_plots,
        )
print('')
print(f'There were {len(list_nunique_1)} features with no variance:')
for a, col in enumerate(list_nunique_1):
    print(f'{a+1} - {col}')

100%|██████████| 47/47 [00:38<00:00,  1.21it/s]


There were 0 features with no variance:


### Move to s3

In [20]:
str_dirname = f'{str_dirname_output}/{str_variant}/liftplots'
list_str_files = os.listdir(str_dirname)
list_str_files = [str_file for str_file in list_str_files if '.png' in str_file]
print(f'There are {len(list_str_files)} files to move to s3:')
for a, str_file in enumerate(list_str_files):
    print(f'{a+1} - {str_file}')

There are 47 files to move to s3:
1 - plt_lift_vehiclemake__app.png
2 - plt_lift_bitdealerapplicantsamezip__app.png
3 - plt_dti__app.png
4 - plt_lift_intservicecontractmileageaddon__app.png
5 - plt_lift_strname__app.png
6 - plt_lift_strdealershiptrackertype__app.png
7 - plt_lift_fltgapinsurance__app.png
8 - plt_lift_ENG-applicationdate__app_quarter.png
9 - plt_fltallowance__app.png
10 - plt_lift_bookvalue__app.png
11 - plt_payment__app.png
12 - plt_lift_fltdowncash__app.png
13 - plt_lift_bitdealertrack__app.png
14 - plt_lift_fltinsuredlifeamount__app.png
15 - plt_lift_dealerstate__app.png
16 - plt_fltdocumentfee__app.png
17 - plt_lift_fltinsureddisabilityamount__app.png
18 - plt_lift_fltapproveddowntotal__app.png
19 - plt_lift_ENG-payment_to_income.png
20 - plt_ENG-loan_to_value.png
21 - plt_lift_intservicecontractterm__app.png
22 - plt_lift_intservicecontractmileage__app.png
23 - plt_lift_bitrouteone__app.png
24 - plt_fltapprovedloantovalue__app.png
25 - plt_lift_bitdealerapplicantsam

In [21]:
%%time

for str_file in tqdm(list_str_files):
    str_local_path = f'{str_dirname}/{str_file}'
    str_bucket_path = f'ad_hoc/get_predictions/{str_variant}/liftplots/{str_file}'
    upload_to_s3(
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project=str_project,
    )
    # rm
    os.remove(str_local_path)

100%|██████████| 47/47 [00:05<00:00,  8.13it/s]

CPU times: user 1.68 s, sys: 56 ms, total: 1.74 s
Wall time: 5.79 s
